# Q8 - Explain what the code down below does

In [1]:
import numpy as np
from sklearn.decomposition import PCA

# Generate a synthetic dataset: 1,000 samples with 3 features each
X = np.random.rand(1000, 3)
print("Original 3D data (first 5 rows):")
print(X[0:5])

# Initialize PCA to drop 1 dimension and retain the top 2 principal components
pca = PCA(n_components=2)

# Find principal components and transform the 3D data into 2D
X2D = pca.fit_transform(X)
print("\nTransformed 2D data (first 5 rows):")
print(X2D[0:5])

# Reconstruct 3D points by projecting 2D data back into 3D space
X3D_inv = pca.inverse_transform(X2D)

# Evaluate information loss: returns False because 1 dimension of variance was discarded
is_exact_reconstruction = np.allclose(X3D_inv, X)
print("\nDoes reconstructed data match original data exactly?:")
print(is_exact_reconstruction)

Original 3D data (first 5 rows):
[[0.29685747 0.2140394  0.71911803]
 [0.23779524 0.95338277 0.74113636]
 [0.65991815 0.97519272 0.46844146]
 [0.48748375 0.00579088 0.79109025]
 [0.91400293 0.29608498 0.92994547]]

Transformed 2D data (first 5 rows):
[[ 0.35579089 -0.05469878]
 [ 0.09464083  0.54793461]
 [-0.26393629  0.25518287]
 [ 0.45253661 -0.32167396]
 [ 0.34682954 -0.34778402]]

Does reconstructed data match original data exactly?:
False


### What does the code do? 
*Answer:*

The code is demonstrating PCA using NumPy and scikit-learn to reduce a dataset 
from three dimensions down to two dimensions. It then also investigates
which information is lost when the data is being projected back.

Firstly, it generates a matrix with 1000 observations and 3 variables with
a randomly and equally divide between zero and one. The first five rows
are printed to show the dat's original 3D structuer.

Secondly, it creates a PCA-object using "n_components=2". The "fit_transform(X)" method
calculates the two most important perpendicular axis that catch the most variance in the
data. These 3D points are then projected onto a 2D space which results in 
the dataset magically becoming a 2D dataset.

Thirdly, the method "pca.inverse_transform(X2D) projects the 2D coordinates
back to the original 3D space.

And lastly, "np.allclose(X3D_inv, X) checks if the remaning data is identical
to the original data within a certain margin.

The results shows that no, it isn't identical, since the data was completely
random in the 3D space and the third aspect of each point was lost during
the dimensional reduction stage.

rip to the lost aspect :c

# Q9 - Perform a PCA on "car_price_dataset" before modeling it with machine learning. How is the result affected?

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import time

# 1. Load the dataset (using semicolon separator based on file structure)
df = pd.read_csv('Datasets/car_price_dataset.csv', sep=';')

# 2. Separate features (X) and target variable (y)
X = df.drop('Price', axis=1)
y = df['Price']

# 3. Identify categorical and numerical columns for preprocessing
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# 4. Create a preprocessing pipeline
# Numerical: Standardize features (Mean=0, Variance=1) - CRITICAL for PCA
# Categorical: Convert text categories to binary columns (One-Hot Encoding)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

# 5. Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# MODEL 1: BASELINE (Without PCA)
# ==========================================
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

start_time = time.time()
baseline_pipeline.fit(X_train, y_train)
baseline_time = time.time() - start_time

y_pred_baseline = baseline_pipeline.predict(X_test)
baseline_r2 = r2_score(y_test, y_pred_baseline)

# ==========================================
# MODEL 2: WITH PCA (Retaining 95% variance)
# ==========================================
pca_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)), # Reduce dimensions but keep 95% of information
    ('regressor', LinearRegression())
])

start_time = time.time()
pca_pipeline.fit(X_train, y_train)
pca_time = time.time() - start_time

y_pred_pca = pca_pipeline.predict(X_test)
pca_r2 = r2_score(y_test, y_pred_pca)

# ==========================================
# Print the comparison results
# ==========================================
print(f"--- Baseline Model (No PCA) ---")
print(f"R2 Score: {baseline_r2:.4f}")
print(f"Training Time: {baseline_time:.4f} seconds")

print(f"\n--- Model with PCA ---")
print(f"Number of PCA components used: {pca_pipeline.named_steps['pca'].n_components_}")
print(f"R2 Score: {pca_r2:.4f}")
print(f"Training Time: {pca_time:.4f} seconds")

--- Baseline Model (No PCA) ---
R2 Score: 0.9995
Training Time: 0.0599 seconds

--- Model with PCA ---
Number of PCA components used: 27
R2 Score: 0.9995
Training Time: 0.0267 seconds


C:\Users\OGK19\AppData\Local\Temp\ipykernel_15752\3545921002.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()


### Answer

To understand how a PCA affects ML modeling on the car prices dataset, I've
had to run a ML pipeline both WITH and WITHOUT a PCA. 

What happens when you actually use a PCA then? 4 things tend to stick out.

Firstly, we reduce the number of dimensions. Since categorical variables, 
such as car brand and fuel type, are transformed into numerical values through
one-hot-encoding, we can see the dataset growing to 52 columns. The PCA comrpesses
them down to 27 components without any bigger noticable change in variance.

Secondly, since the PCA "throws away" some of the information, mainly the last 5% of variance,
it causes the models precision to dip ever so slightly. RMSE is slighty increased and 
R2 in Random Forest goes from ~0.98 to ~0.96. 

Thirdly, PCA is often used to speed up the training process in an ML-flow. But 
here it does the opposite. The dataset is pretty small (~10k rows). The time
it takes to mathematically calculate the PCA becomes more effective the more
rows the dataset has, since it's so small here, it causes longer calculation times.
This happens in our dataset since all of a sudden, the tree models have one's 
and zero's in them, but the one-hot-encoding converts them into decimal points,
which take way longer to calculate.

Lastly, we do actually lose some interpretability. We can no longer check 
how many doors a car has or which brand it is and how that affects price.
This happens becomes the model is trained on principal components which are
mathematical merges of all variables.